# SIH26184: Predictive Cash-Withdrawal Location Intelligence
## Notebook 1: Data Understanding, EDA, Feature Engineering & Baseline Models
---
**Theme:** Blockchain & Cybersecurity | **Organization:** Ministry of Home Affairs - I4C

This notebook documents:
1. Raw dataset schema and data inspection (`indian_banking_transactions.csv`, RBI ATM registry)
2. Exploratory Data Analysis (EDA) on cyber fraud timing, channels, and geographical distribution
3. Deterministic Candidate Location Generation & Coverage Evaluation
4. Leakage-free Feature Pipeline across 8 feature families
5. Evaluation of Progressive Baseline Models:
   - Baseline 0: Historical Hotspot / Frequency Heuristic
   - Baseline 1: Spatial Proximity Heuristic
   - Baseline 2: Regularized Logistic Regression
   - Baseline 3: Random Forest Classifier


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import json
from src.data.atm_processor import haversine_distance_km
from src.features.feature_pipeline import FeaturePipeline
from src.candidate_generation.candidate_generator import CandidateGenerator
from src.models.baselines import run_all_baselines
from src.evaluation.metrics import format_metrics_table


### 1. Inspect Grounded Candidate Dataset & Chronological Splits

In [ ]:
df = pd.read_parquet("../data/synthetic/candidate_dataset.parquet")
print(f"Total candidate-level rows: {len(df)}")
print("\n--- Split Summary ---")
print(df.groupby("split")["case_id"].nunique().rename("unique_cases"))
print(df.groupby("split")["target"].agg(["count", "sum", "mean"]).rename(columns={"count": "rows", "sum": "positives", "mean": "pos_rate"}))


### 2. Candidate Generator Coverage Analysis

In [ ]:
gen = CandidateGenerator(atms_parquet_path="../data/processed/atms_indexed.parquet")
with open("../data/synthetic/grounded_cases.json", "r", encoding="utf-8") as f:
    cases = json.load(f)
test_cases = [c for c in cases if c["prediction_time"] > "2023-06-30T23:59:59"]
coverage_metrics = gen.evaluate_coverage(test_cases, max_candidates=25)
print("Test Set Candidate Generator Coverage:", coverage_metrics)


### 3. Progressive Baselines Evaluation

In [ ]:
train_df = df[df["split"] == "train"]
test_df = df[df["split"] == "test"]
pipeline = FeaturePipeline.load(artifact_dir="../artifacts/model")
baseline_results = run_all_baselines(train_df, test_df, pipeline)
print("\n--- Baseline Comparison on Out-Of-Time Test Set ---")
print(format_metrics_table(baseline_results))
